In [ ]:
import pandas as pd
import pickle
import numpy as np
import torch

import plotly.express as px
import matplotlib.pyplot as plt

import sys
sys.path.append('..')

from configs.split_config import TG_SUPERPOP_DICT, ethnic_background_name_map

In [ ]:
# Arguments for projecting into TG PC space with plink

# plink2 --extract variants --out ukb_1kg_projections --pfile /gpfs/gpfs0/ukb_data/plink/plink --read-freq tg_pca.acount --score tg_pca.eigenvec.allele 2 5 variance-standardize --score-col-nums 6-25

In [ ]:
UKB_TG_PROJECTIONS_PATH = '/trinity/home/s.mishra/tg_pca/ukb_1kg_projections.sscore'

TG_ANCESTRY_MODEL_PATH = '/trinity/home/s.mishra/tg_pca/tg_pca.pkl'
SUPERPOPULATIONS_OUTPUT_PATH = '/trinity/home/s.mishra/uk-biobank/superpopulations.csv'

In [ ]:
df_tg = pd.read_table('/trinity/home/s.mishra/tg_pca/tg_pca.sscore')
df_tg = df_tg.drop(['ALLELE_CT', 'NAMED_ALLELE_DOSAGE_SUM'], axis=1)
df_tg['pop'] = 'tg'
df_ukb = pd.read_table(UKB_TG_PROJECTIONS_PATH)
df_ukb = df_ukb.drop(['#FID', 'ALLELE_CT', 'NAMED_ALLELE_DOSAGE_SUM'], axis=1)[::10]
df_ukb['pop'] = 'ukb'
df_ukb.columns = df_tg.columns
df = pd.concat([df_ukb, df_tg])
px.scatter(df, x='PC1_AVG', y='PC2_AVG', color='pop')

In [ ]:
superpopulations_node_map = {
    'EUR': 0,
    'SAS': 1,
    'AFR': 2,
    'EAS': 3,
    'AMR': 4
}

In [ ]:
ukb_tg_projections = pd.read_table(UKB_TG_PROJECTIONS_PATH)
X = ukb_tg_projections.filter(like="_AVG").values
ancestry_model = pickle.load(open(TG_ANCESTRY_MODEL_PATH, 'rb'))
ancestry_model.eval()
pred_probs = torch.softmax(ancestry_model.forward(torch.Tensor(X)), dim=1).detach().numpy()

populations = list(sorted(TG_SUPERPOP_DICT.keys()))
ukb_tg_projections['pred_ancestry'] = np.vectorize(lambda index: populations[index])(np.argmax(pred_probs, axis=1))
ukb_tg_projections['pred_superpop'] = ukb_tg_projections.pred_ancestry.map(TG_SUPERPOP_DICT)
ukb_tg_projections['node_index'] = ukb_tg_projections.pred_superpop.map(superpopulations_node_map)

superpop_list = np.vectorize(lambda x: superpopulations_node_map[TG_SUPERPOP_DICT[x]])(populations)
superpop_probs = np.zeros((ukb_tg_projections.shape[0], len(superpopulations_node_map)))
for i in range(len(superpopulations_node_map)):
    superpop_probs[:, i] = np.sum(pred_probs[:, superpop_list == i], axis=1)
    
ukb_tg_projections['superpop_confidence'] = np.max(superpop_probs, axis=1)

ukb_tg_projections.loc[ukb_tg_projections.superpop_confidence > 0.95,
                       ['IID', 'node_index']].to_csv(SUPERPOPULATIONS_OUTPUT_PATH, index=False)

In [ ]:
ukb_tg_projections.pred_ancestry.value_counts()

In [ ]:
ukb_tg_projections.pred_superpop.value_counts()

In [ ]:
px.scatter(ukb_tg_projections, x='SCORE1_AVG', y='SCORE2_AVG', color='pred_ancestry').write_html('ancestries_pc1v2.html')
px.scatter(ukb_tg_projections, x='SCORE3_AVG', y='SCORE4_AVG', color='pred_ancestry').write_html('ancestries_pc3v4.html')
px.scatter(ukb_tg_projections, x='SCORE3_AVG', y='SCORE4_AVG', color='pred_superpop').write_html('superpopulations_pc3v4.html')
px.scatter(ukb_tg_projections, x='SCORE1_AVG', y='SCORE2_AVG', color='pred_superpop').write_html('superpopulations_pc1v2.html')

In [ ]:
from preprocess.splitter import SplitBase

df = SplitBase().get_ethnic_background()
df.index = df.IID
ukb_tg_projections.index =  ukb_tg_projections.IID
ukb_tg_projections['sr_ancestry_code'] = df.ethnic_background

In [ ]:
for code in ukb_tg_projections.sr_ancestry_code.unique():
    print(f"EB code: {code}")
    if code in ethnic_background_name_map:
        print(f"Ethnicity: {ethnic_background_name_map[code]}")
    print("Value counts:")    
    print(ukb_tg_projections.loc[ukb_tg_projections.sr_ancestry_code == code].pred_ancestry.value_counts())
    print("\n")